## Synergy of mergerd Kabuki 1 & 2 syndrome cohorts

In [1]:
from ppktstore.registry import configure_phenopacket_registry
import pandas as pd
from phenosign import PhenotypeDatasetBuilder, HPOCorrelationAnalyzer, SynergyAnalyzer

In [2]:
registry = configure_phenopacket_registry()
with registry.open_phenopacket_store() as ps:
    kmt2d = list(ps.iter_cohort_phenopackets("ITPR1"))
    kdm6a = list(ps.iter_cohort_phenopackets("GLI3"))

phenopackets = kmt2d + kdm6a
print(f"KMT2D={len(kmt2d)}, KDM6A={len(kdm6a)}, combined={len(phenopackets)}")

KMT2D=170, KDM6A=81, combined=251


In [3]:
dataset = PhenotypeDatasetBuilder(phenopackets).build(build_gpsea_cohort=False)

# Inspect what's actually in the merged data before hardcoding IDs:
diseases_df, sex_df, genes_df, _ = dataset.describe_conditions()
print(diseases_df)  

                                                         label n_individuals
disease_id                                                                  
OMIM:117360  Spinocerebellar ataxia 29, congenital nonprogr...   104 (41.4%)
OMIM:175700               Greig cephalopolysyndactyly syndrome    50 (19.9%)
OMIM:206700                                 Gillespie syndrome    39 (15.5%)
OMIM:606658                          Spinocerebellar ataxia 15    27 (10.8%)
OMIM:146510                            Pallister-Hall syndrome     21 (8.4%)
OMIM:174200             Polydactyly, postaxial, types A1 and B     10 (4.0%)


In [4]:
analyzer = HPOCorrelationAnalyzer(dataset=dataset)

results = analyzer.compute_correlation_matrix()

results.results_table.head()

Calculating pairwise correlation:   0%|          | 0/772 [00:00<?, ?it/s]

Calculating pairwise correlation: 100%|██████████| 772/772 [00:01<00:00, 480.70it/s]


,HPO_A,HPO_A_label,HPO_B,HPO_B_label,correlation,p_value,adj_p_value,n(A:E/B:E),n(A:E/B:O),n(A:O/B:E),n(A:O/B:O),n_individuals,n_pmids,pmids
502,HP:0025336,Delayed ability to sit,HP:0031936,Delayed ability to walk,0.968974,1.933442e-17,9.918555e-15,27,1,0,37,65,11,20082166;21367767;22986007;25981959;27062503;2...
133,HP:0000750,Delayed speech and language development,HP:0002194,Delayed gross motor development,0.916698,4.180318e-17,1.072252e-14,25,3,0,48,76,15,15623688;20082166;21367767;22986007;25981959;2...
144,HP:0000750,Delayed speech and language development,HP:0031936,Delayed ability to walk,0.916147,8.525824e-17,1.093437e-14,26,3,0,44,73,15,15623688;20082166;21367767;22986007;25981959;2...
143,HP:0000750,Delayed speech and language development,HP:0025336,Delayed ability to sit,0.967955,7.559320e-17,1.093437e-14,26,0,1,36,63,8,20082166;21367767;22986007;27108798;32496011;3...
126,HP:0000750,Delayed speech and language development,HP:0001270,Motor delay,0.868744,8.344349e-16,8.561302e-14,25,5,0,48,78,16,15623688;20082166;21367767;22986007;25981959;2...


In [5]:
results.save_correlation_results(
    output_file="../case_studies/ITPR1_GLI3/correlation_results_ITPR1_GLI3.csv"
)

In [6]:
results.plot_correlation_heatmap_with_significance(
    title_name="Cohort FBN1",
) 

In [7]:
results.save_correlation_heatmap()

### **Synergy Analysis Between HPO Terms and Target Variables**

In [8]:
diseases_df, sex_df, genes_df, variant_effects_df = dataset.describe_conditions()

diseases_df

,label,n_individuals
disease_id,,
OMIM:117360,"Spinocerebellar ataxia 29, congenital nonprogr...",104 (41.4%)
OMIM:175700,Greig cephalopolysyndactyly syndrome,50 (19.9%)
OMIM:206700,Gillespie syndrome,39 (15.5%)
OMIM:606658,Spinocerebellar ataxia 15,27 (10.8%)
OMIM:146510,Pallister-Hall syndrome,21 (8.4%)
OMIM:174200,"Polydactyly, postaxial, types A1 and B",10 (4.0%)


In [9]:
sex_df

,n_individuals
sex,
female,68 (27.1%)
male,48 (19.1%)
unknown,135 (53.8%)


In [10]:
genes_df

,n_individuals
gene_symbol,
ITPR1,170 (67.7%)
GLI3,81 (32.3%)


In [11]:
variant_effects_df

In [12]:
synergy_analyzer = SynergyAnalyzer(dataset=dataset)

In [13]:
from phenosign import has_gene

condition = dataset.get_condition(
    has_gene("GLI3"),                       # 1 = GLI3 patient, 0 = ITPR1 patient
    name="GLI3 vs ITPR1",
)

synergy_analyzer = SynergyAnalyzer(dataset=dataset)
results = synergy_analyzer.compute_synergy_matrix(condition=condition)
results.results_table.head()
results.plot_synergy_heatmap()

Calculating pairwise synergy: 100%|██████████| 361/361 [00:00<00:00, 3148.80it/s]
